# 02 — 3D CNN classification: grade LGG / HGG + radiological pseudo-profile

Pseudo-profiles (LGG-DA / LGG-OLIGO / HGG-ANA / HGG-GBM) are **K-Means clusters of tumor volumes**, not
histological subtypes. The trained model is saved as `models/classification/cnn3d_grade_profile.pt`.

In [ ]:
# --- Setup: mount Drive + install the BrainTumorLab code (run this cell first) ---
from google.colab import drive
drive.mount("/content/drive")

import os, sys
REPO_URL = ""   # e.g. "https://github.com/<your-account>/BrainTumorLab.git" ; empty = use the zip on Drive
ZIP_ON_DRIVE = "/content/drive/MyDrive/miniProjetAI/BrainTumorLab.zip"
if not os.path.isfile("/content/BrainTumorLab/pyproject.toml"):
    import shutil
    shutil.rmtree("/content/BrainTumorLab", ignore_errors=True)
    if REPO_URL:
        !git clone -q {REPO_URL} /content/BrainTumorLab
    else:
        !rm -rf /content/btl_tmp && unzip -q -o "{ZIP_ON_DRIVE}" -d /content/btl_tmp
        _src = "/content/btl_tmp/BrainTumorLab" if os.path.isfile("/content/btl_tmp/BrainTumorLab/pyproject.toml") else "/content/btl_tmp"
        shutil.move(_src, "/content/BrainTumorLab")
!pip install -q -e /content/BrainTumorLab
sys.path.insert(0, "/content/BrainTumorLab/src")

import torch
from braintumorlab.config import Paths, seed_everything
seed_everything(0)
paths = Paths("/content/drive/MyDrive/miniProjetAI")
DATA_ROOT = paths.ensure_data_extracted()     # unzips BraTS_2020_Data_preprocessing_nii_FIXED.zip once
print("data:", DATA_ROOT, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

In [ ]:
from braintumorlab import classification as C
from braintumorlab.analysis import volumes_table
import pandas as pd, os
vol_csv = paths.base / "recap_stats_volumes_2020_v2.csv"       # written by notebook 00, part A
volumes = pd.read_csv(vol_csv) if vol_csv.exists() else volumes_table(DATA_ROOT)
df = C.build_patient_table(paths.metadata_csv, volumes)
df = C.add_pseudo_profiles(df)            # K-Means fitted on the train split only
train_df, val_df, test_df = C.split_tables(df)
print(len(train_df), len(val_df), len(test_df))
df.groupby(["grade_str", "pseudo_profile"]).size()

In [ ]:
MODALITIES = ["FLAIR", "T1ce", "T2"]
DOWNSAMPLE = 4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
file_index = C.build_file_index(DATA_ROOT, MODALITIES)
datasets = [C.BraTS3DClassificationDataset(d, file_index, MODALITIES, DOWNSAMPLE) for d in (train_df, val_df, test_df)]
BALANCE = "sampler"                       # "sampler" OR "weights", never both
train_loader, val_loader, test_loader = C.make_loaders(*datasets, batch_size=2, balance=BALANCE)
crit_g, crit_s = C.make_criteria(train_df, balance=BALANCE, device=device)
model = C.MultiTaskGliomaClassifier(in_channels=len(MODALITIES)).to(device)

In [ ]:
CLF_PATH = str(paths.base / "models" / "classification" / "cnn3d_grade_profile.pt")
hist = C.train_classifier(model, train_loader, val_loader, crit_g, crit_s, device, epochs=60, out_path=CLF_PATH)
hist.plot(x="epoch", y=["train_loss", "val_loss"]); hist.plot(x="epoch", y=["val_grade_bal_acc", "val_subtype_bal_acc"])

In [ ]:
import json
model = C.load_classifier(CLF_PATH, device)
res = C.run_epoch(model, test_loader, crit_g, crit_s, device)
report = C.classification_report_dict(res)
report.update({"model": "cnn3d", "split": "test", "modalities": MODALITIES})
os.makedirs(paths.eval_dir, exist_ok=True)
json.dump(report, open(paths.eval_dir / "cls_cnn3d.json", "w"), indent=1)   # shown in the app
report

In [ ]:
pid = test_df["patient_id"][0]
C.predict_grade(model, file_index[pid], MODALITIES, DOWNSAMPLE, device=device)

### Use the classifier in the app
Notebook 05 loads `models/classification/cnn3d_grade_profile.pt` automatically when it exists.